In [ ]:
!pip install -q "ultralytics>=8.0.0" timm supervision opencv-python-headless
!pip install ultralytics flask flask-ngrok pyngrok
!pip install flask-cors pyngrok ultralytics
!pip install websockets nest_asyncio pyngrok ultralytics
!pip install flask flask-socketio pyngrok opencv-python-headless ultralytics timm


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.9/41.9 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 33.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 217.4/217.4 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.1/82.1 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.8/59.8 kB 5.7 MB/s eta 0:00:00


In [ ]:
#FOR TENSOR RT FORMAT ONLY
!pip install tensorrt tensorrt-cu12 ultralytics pyngrok flask-cors

In [ ]:
import torch
import torch.nn as nn
import timm
import cv2
import numpy as np
import json
import os
import pandas as pd
import threading
from flask import Flask
from flask_socketio import SocketIO, emit
from pyngrok import ngrok
from ultralytics import YOLO
from google.colab import drive

# 1. SETUP & INITIALIZATION
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
drive.mount('/content/drive', force_remount=True)

# CALIBRATED SETTINGS
E_MIN = 0.001231
E_MAX = 0.015466
BLEMISH_THRESHOLD = 0.08
SAVE_DIR = '/content/drive/MyDrive/SmartShelf_Outputs'
os.makedirs(SAVE_DIR, exist_ok=True)

class SmartShelfAutoencoder(nn.Module):
    def __init__(self, model_name='resnet50'):
        super().__init__()

        self.encoder = timm.create_model(model_name, pretrained=True, features_only=True)
        self.decoder = nn.Sequential(

            nn.ConvTranspose2d(2048, 512, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(512, 256, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(64, 3, kernel_size=4, stride=2, padding=1),
            nn.Sigmoid()
        )

    def forward(self, x):
        features = self.encoder(x)
        latent_space = features[-1]
        return self.decoder(latent_space)

# Load YOLO and Autoencoders
yolo_model = YOLO('/content/drive/MyDrive/AI_MODELS/smartshelf_best.pt')
ae_models = {}
for fruit in ['PEAR']:
    path = f'/content/drive/MyDrive/WEIGHTS/{fruit.lower()}_ae.pth'
    if os.path.exists(path):
        m = SmartShelfAutoencoder().to(device)
        m.load_state_dict(torch.load(path))
        m.eval()
        ae_models[fruit] = m

# 2. CORE PIPELINE
def run_full_smartshelf_process(img):
    results = yolo_model(img, verbose=False)[0]

    if not results.boxes or not results.masks:
        return json.dumps({"status": "error", "message": "No fruit or mask detected"})

    box = results.boxes[0]
    label = yolo_model.names[int(box.cls[0])].upper()
    coords = box.xyxy[0].cpu().numpy().astype(int)

    raw_mask = (results.masks.data[0].cpu().numpy() > 0.5).astype(np.uint8)
    raw_mask = cv2.resize(raw_mask, (img.shape[1], img.shape[0]), interpolation=cv2.INTER_NEAREST)

    soft_mask = cv2.GaussianBlur(raw_mask.astype(float), (15, 15), 0)

    masked_img = (img.astype(float) * np.stack([soft_mask] * 3, axis=-1)).astype(np.uint8)

    mask_crop = raw_mask[coords[1]:coords[3], coords[0]:coords[2]]
    mask_resized = cv2.resize(mask_crop, (640, 640), interpolation=cv2.INTER_NEAREST)
    mask_tensor = torch.tensor(mask_resized).to(device).float()

    crop = masked_img[coords[1]:coords[3], coords[0]:coords[2]]
    crop_input = cv2.resize(crop, (640, 640), interpolation=cv2.INTER_LINEAR)

    crop_rgb = cv2.cvtColor(crop_input, cv2.COLOR_BGR2RGB)

    if label not in ae_models:
        return json.dumps({"status": "error", "message": f"No AE for {label}"})

    model = ae_models[label]
    tensor_in = torch.tensor(crop_rgb).permute(2, 0, 1).float().unsqueeze(0).to(device) / 255.0

    with torch.no_grad():
        recon = model(tensor_in)

    # IMAGE SAVING BLOCK
    timestamp_str = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")

    yolo_save_path = os.path.join(SAVE_DIR, f"{label}_{timestamp_str}_yolo.jpg")
    cv2.imwrite(yolo_save_path, crop_input)

    recon_np = recon.squeeze(0).cpu().permute(1, 2, 0).numpy()
    recon_output = (recon_np * 255.0).astype(np.uint8)

    recon_save_path = os.path.join(SAVE_DIR, f"{label}_{timestamp_str}_recon.jpg")
    cv2.imwrite(recon_save_path, recon_output)

    # METRIC 1: MASKED MSE (Intensity)
    sq_error = torch.pow(recon.squeeze(0) - tensor_in.squeeze(0), 2)
    mse_map = torch.mean(sq_error, dim=0)
    fruit_pixels_mse = mse_map * mask_tensor
    num_fruit_pixels = torch.sum(mask_tensor)

    if num_fruit_pixels == 0:
        masked_mse = 0.0
        blemish_pct = 0.0
    else:
        masked_mse = (torch.sum(fruit_pixels_mse) / num_fruit_pixels).item()

        # METRIC 2: BLEMISH PERCENTAGE (Area Coverage)
        abs_error = torch.abs(recon.squeeze(0) - tensor_in.squeeze(0))
        diff_map = torch.mean(abs_error, dim=0)
        fruit_pixels_diff = diff_map * mask_tensor

        blemish_mask = (fruit_pixels_diff > BLEMISH_THRESHOLD).float()
        num_blemish_pixels = torch.sum(blemish_mask)
        blemish_pct = (num_blemish_pixels / num_fruit_pixels).item() * 100

    final_data = {
        "label": label,
        "masked_mse_intensity": round(float(masked_mse), 8),
        "blemish_area_percentage": round(float(blemish_pct), 2),
        "fruit_pixel_count": int(num_fruit_pixels.item()),
        "status": "Fresh" if masked_mse < 0.005 else "Spoilage Detected",
        "timestamp": str(pd.Timestamp.now())
    }

    return json.dumps(final_data)


# 3. WEBSOCKET LISTENER
app = Flask(__name__)
socketio = SocketIO(app, cors_allowed_origins="*", async_mode='threading', max_http_buffer_size=10000000)

@socketio.on('connect')
def handle_connect():
    print("New camera client connected via WebSocket.")

@socketio.on('disconnect')
def handle_disconnect():
    print("Camera client disconnected.")

@socketio.on('process_frame')
def handle_frame(file_bytes):
    npimg = np.frombuffer(file_bytes, np.uint8)
    img = cv2.imdecode(npimg, cv2.IMREAD_COLOR)

    if img is not None:
        analysis_results = run_full_smartshelf_process(img)
        emit('analysis_result', analysis_results)
    else:
        emit('analysis_result', json.dumps({"status": "error", "message": "Failed to decode image buffer"}))

ngrok.set_auth_token("3ASqrcPQ7WGZeYryJZnuRu7zcoK_7vgmtG5DHyZfPHArEc8DX")
public_url = ngrok.connect(5005)

print(f"YOUR WEBSOCKET SERVER URL IS:")
print(f"{public_url.public_url}")
print("Configure your local Python camera script to use a Socket.IO client and connect to this URL.")

socketio.run(app, host='0.0.0.0', port=5005, allow_unsafe_werkzeug=True)

Mounted at /content/drive


YOUR WEBSOCKET SERVER URL IS:
https://superprecise-lisha-unwelded.ngrok-free.dev
Configure your local Python camera script to use a Socket.IO client and connect to this URL.
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5005
 * Running on http://172.28.0.12:5005
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [02/Apr/2026 11:53:29] "GET /socket.io/?transport=polling&EIO=4&t=1775130808.7775319 HTTP/1.1" 200 -


New camera client connected via WebSocket.


INFO:werkzeug:127.0.0.1 - - [02/Apr/2026 11:57:29] "GET /socket.io/?transport=websocket&EIO=4&sid=w64BOQrH1ZSu2UARAAAA&t=1775130809.9896593 HTTP/1.1" 200 -


Camera client disconnected.


INFO:werkzeug:127.0.0.1 - - [02/Apr/2026 12:03:38] "GET /socket.io/?transport=polling&EIO=4&t=1775131418.0454564 HTTP/1.1" 200 -


New camera client connected via WebSocket.


INFO:werkzeug:127.0.0.1 - - [02/Apr/2026 12:09:02] "GET /socket.io/?transport=websocket&EIO=4&sid=YBmVJkjYmbdpDvMCAAAC&t=1775131419.060935 HTTP/1.1" 200 -


Camera client disconnected.
